# Geometric Mean Sanity Tests

This notebook tests the phase-aligned complex geometric mean. The paper-level invariant used here is the local geometric-mean step: for object patches, `(z_GM)^2 = z_current * z_MG`; for shifted probe patches, `(Q_GM)^2 = Q_current * Q_plus`. The square-root branch is selected to stay aligned with the current iterate.


In [ ]:
import os
from pathlib import Path
import sys

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/magpie-matplotlib")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src" / "common.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

%load_ext autoreload
%autoreload 2


import torch
import matplotlib.pyplot as plt

from algorithms.geometric_mean import aligned_geom_mean_torch

torch.manual_seed(0);


In [ ]:
def random_complex(shape, dtype=torch.complex64):
    real_dtype = torch.float64 if dtype == torch.complex128 else torch.float32
    return torch.randn(shape, dtype=real_dtype) + 1j * torch.randn(shape, dtype=real_dtype)


def branch_is_aligned(x, anchor, tol=1e-6):
    anchor = anchor.to(x.dtype)
    inner = x * torch.conj(anchor)
    re = inner.real
    im = inner.imag
    return bool(((re > -tol) & ((re.abs() > tol) | (im >= -tol))).all())


def assert_close(name, actual, expected, atol=1e-6, rtol=1e-6):
    err = torch.max(torch.abs(actual - expected)).item()
    scale = torch.max(torch.abs(expected)).item()
    allowed = atol + rtol * scale
    print(f"{name}: max error={err:.3e}, allowed={allowed:.3e}")
    assert err <= allowed


## Test 1: Algebra and Branch Selection

The returned value should square to `a * b`, and its branch should be aligned to `anchor`: prefer positive real inner product with the anchor, with a deterministic imaginary-part tie break.


In [ ]:
a = random_complex((8, 16, 16))
b = random_complex((8, 16, 16))
anchor = random_complex((8, 16, 16))

x = aligned_geom_mean_torch(a, b, anchor)
assert_close("x**2 = a*b", x * x, a * b, atol=2e-5, rtol=2e-5)
assert branch_is_aligned(x, anchor)
print("branch alignment: passed")


## Test 2: MAGPIE-GM Local Object Formula

For a small local perturbation `z_MG = z_current * amp * exp(i delta)`, the aligned geometric mean should be `z_current * sqrt(amp) * exp(i delta / 2)`. This is the local object-patch relation `(z_GM)^2 = z_current * z_MG` with the branch aligned to `z_current`.


In [ ]:
z_current = random_complex((4, 1, 32, 32))
delta = 0.75 * torch.pi * (2 * torch.rand((4, 1, 32, 32)) - 1)
amp = 0.25 + 2.0 * torch.rand((4, 1, 32, 32))
z_mg = z_current * amp * torch.exp(1j * delta)

z_gm = aligned_geom_mean_torch(z_current, z_mg, z_current)
expected = z_current * torch.sqrt(amp) * torch.exp(0.5j * delta)

assert_close("object GM formula", z_gm, expected, atol=2e-5, rtol=2e-5)
assert_close("object GM square", z_gm * z_gm, z_current * z_mg, atol=2e-5, rtol=2e-5)
print("local object MAGPIE-GM relation: passed")


## Test 3: Shifted-Probe Formula

The probe update uses the same operation in the shifted-probe frame before adjoint shifting: `(Q_GM)^2 = Q_current * Q_plus`, with the branch aligned to `Q_current`.


In [ ]:
q_current = random_complex((5, 1, 32, 32))
delta_q = 0.6 * torch.pi * (2 * torch.rand((5, 1, 32, 32)) - 1)
amp_q = 0.5 + 1.5 * torch.rand((5, 1, 32, 32))
q_plus = q_current * amp_q * torch.exp(1j * delta_q)

q_gm = aligned_geom_mean_torch(q_current, q_plus, q_current)
expected_q = q_current * torch.sqrt(amp_q) * torch.exp(0.5j * delta_q)

assert_close("probe GM formula", q_gm, expected_q, atol=2e-5, rtol=2e-5)
assert_close("probe GM square", q_gm * q_gm, q_current * q_plus, atol=2e-5, rtol=2e-5)
print("shifted-probe MAGPIE-GM relation: passed")


## Test 4: Explicit Branch and Tie Cases

These examples check the nontrivial sign flip and the deterministic tie break when `Re(x * conj(anchor))` is near zero.


In [ ]:
case_a = torch.tensor([1, 1, 1, 1, 0], dtype=torch.complex64)
case_b = torch.tensor([1, 1, -1, -1, 3], dtype=torch.complex64)
case_anchor = torch.tensor([1, -1, 1, -1, 1], dtype=torch.complex64)
case_expected = torch.tensor([1, -1, 1j, -1j, 0], dtype=torch.complex64)

case_x = aligned_geom_mean_torch(case_a, case_b, case_anchor)
assert_close("explicit branch cases", case_x, case_expected)
print(case_x)


## Test 5: Broadcasting and Dtype Promotion

The helper should work on minibatch-shaped tensors and should promote to complex128 if any input is float64 or complex128.


In [ ]:
a_broadcast = random_complex((2, 3), dtype=torch.complex64)
b_broadcast = random_complex((1, 3), dtype=torch.complex128)
anchor_broadcast = random_complex((2, 1), dtype=torch.complex64)

x_broadcast = aligned_geom_mean_torch(a_broadcast, b_broadcast, anchor_broadcast)
print("shape:", tuple(x_broadcast.shape))
print("dtype:", x_broadcast.dtype)
assert x_broadcast.shape == (2, 3)
assert x_broadcast.dtype == torch.complex128
assert_close("broadcast square", x_broadcast * x_broadcast, a_broadcast * b_broadcast, atol=1e-12, rtol=1e-12)


## Phase Sweep

The phase sweep shows the branch choice. For anchor `1` and `b = exp(i delta)`, the chosen geometric mean follows half the phase difference and flips branch to remain aligned with the anchor.


In [ ]:
delta_sweep = torch.linspace(-2 * torch.pi, 2 * torch.pi, 801)
a_sweep = torch.ones_like(delta_sweep, dtype=torch.complex64)
b_sweep = torch.exp(1j * delta_sweep)
anchor_sweep = torch.ones_like(delta_sweep, dtype=torch.complex64)
x_sweep = aligned_geom_mean_torch(a_sweep, b_sweep, anchor_sweep)

plt.figure(figsize=(7, 4))
plt.plot(delta_sweep.numpy(), torch.angle(x_sweep).numpy(), lw=1.6)
plt.axhline(0, color="k", lw=0.8, alpha=0.5)
plt.xlabel("phase difference delta [rad]")
plt.ylabel("phase of aligned geometric mean [rad]")
plt.grid(True, ls=":", alpha=0.6)
plt.tight_layout()
plt.show()

square_error = torch.max(torch.abs(x_sweep * x_sweep - a_sweep * b_sweep)).item()
print(f"phase-sweep square error: {square_error:.3e}")
